# 1.3 Neural Networks with PyTorch

**KEY CONCEPTS to define:**
- **PyTorch**: Facebook's deep learning framework - the tool we'll use to build neural networks
- **Tensor**: PyTorch's version of an array - the fundamental data structure for deep learning
- **Device**: Where computations happen - CPU (slower) or GPU (faster)


In [ ]:
# Core scientific computing
import numpy as np
import matplotlib.pyplot as plt

# Deep learning with PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Detect available computing device (GPU or CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = "NVIDIA GPU"
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon GPU"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"✓ Using device: {device_name}")
print(f"  PyTorch will run computations on: {device}")

### 1. Single Convolution Filter

In [ ]:
# Create a 15×15 test image with simple patterns
img = torch.zeros(15, 15)
img[:, 7] = 1.0              # Vertical line in center
img[3:6, 10:13] = 1.0        # Top-right square
img[9:12, 2:5] = 1.0         # Bottom-left square

# Reshape for PyTorch: (batch_size, channels, height, width)
img_tensor = img.unsqueeze(0).unsqueeze(0)  # (1, 1, 15, 15)

f, ax = plt.subplots(1,1,figsize=(3,3))
ax.imshow(img, cmap='gray', vmin=0, vmax=1)
ax.set_title('Input Image\n15×15 pixels', fontweight='bold', fontsize=12)
ax.axis('off')

In [ ]:
# Create a convolution layer: 1 input channel → 1 filter → 1 output channel
conv_single = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)

# Manually set filter weights to detect vertical edges
with torch.no_grad():  # Don't track gradients for manual weight setting
    conv_single.weight[0, 0] = torch.tensor([
        [-1,  0,  1],   # Left side negative
        [-1,  0,  1],   # Middle neutral
        [-1,  0,  1]    # Right side positive
    ], dtype=torch.float32)


f, ax = plt.subplots(1,1,figsize=(3,3))
ax.imshow(conv_single.weight[0, 0].detach(), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Vertical Edge Filter\n3×3 weights', fontweight='bold', fontsize=12)
ax.axis('off')

# Add text annotations to filter
for i in range(3):
    for j in range(3):
        value = conv_single.weight[0, 0, i, j].item()
        ax.text(j, i, f'{value:.0f}', ha='center', va='center', 
                    color='white' if abs(value) > 0.5 else 'black', fontsize=14, fontweight='bold')


In [ ]:
# Apply the filter to the image
output_single = conv_single(img_tensor)

f, ax = plt.subplots(1,1,figsize=(3,3))
ax.imshow(output_single.squeeze().detach(), cmap='viridis')
ax.set_title('Output Feature Map\n(Vertical edges detected)', fontweight='bold', fontsize=12)
ax.axis('off')

**KEY CONCEPTS to define:**
- **Convolution**: A mathematical operation that slides a small filter across an image
- **Filter (or Kernel)**: A small matrix of weights (e.g., 3×3) that detects specific patterns
- **Feature map**: The output after applying a filter - shows where patterns were detected
- **Edge detector**: A filter designed to find transitions from dark to light
- **Channel**: A layer of information (grayscale = 1 channel, RGB = 3 channels)


### 2. Multiple Filters 

**KEY CONCEPTS:**
- **Multiple filters**: 16 pattern detectors working simultaneously
- **Conv2d(1, 16)**: 1 input channel → 16 filters → 16 output channels


In [ ]:
# Create convolution layer with 16 filters (instead of 1)
conv_multi = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)

# Apply all 16 filters to the image
output_multi = conv_multi(img_tensor)


In [ ]:
print("Multiple Filters in Parallel:")
print(f"  Input:  {img_tensor.shape}  → 1 channel")
print(f"  Output: {output_multi.shape} → 16 channels!")
print(f"\n  16 filters process the same input simultaneously")
print(f"  Each filter creates one output channel")

In [ ]:
# Extract weights and outputs from Filter 1 and Filter 9
filter_1_weights = conv_multi.weight[0, 0].detach()
filter_9_weights = conv_multi.weight[8, 0].detach()

output_filter_1 = output_multi[0, 0].detach()
output_filter_9 = output_multi[0, 8].detach()


In [ ]:
# Create 2-row comparison: Filter 1 (top) vs Filter 9 (bottom)
fig, axes = plt.subplots(2, 3, figsize=(10, 5))

# ===== ROW 1: Filter 1 =====
axes[0, 0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Input Image', fontweight='bold', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(filter_1_weights, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0, 1].set_title('Filter 1 Weights\n(3×3 matrix)', fontweight='bold', fontsize=11)
axes[0, 1].axis('off')

im1 = axes[0, 2].imshow(output_filter_1, cmap='viridis')
axes[0, 2].set_title('Filter 1 Response\n(What it detected)', fontweight='bold', fontsize=11)
axes[0, 2].axis('off')
plt.colorbar(im1, ax=axes[0, 2], fraction=0.046)

# ===== ROW 2: Filter 9 =====
axes[1, 0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Input Image\n(same)', fontweight='bold', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(filter_9_weights, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1, 1].set_title('Filter 9 Weights\n(different!)', fontweight='bold', fontsize=11)
axes[1, 1].axis('off')

im9 = axes[1, 2].imshow(output_filter_9, cmap='viridis')
axes[1, 2].set_title('Filter 9 Response\n(different pattern!)', fontweight='bold', fontsize=11)
axes[1, 2].axis('off')
plt.colorbar(im9, ax=axes[1, 2], fraction=0.046)

plt.suptitle('Different Weights → Different Patterns Detected', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Encoder Block (Conv + ReLU + MaxPool)

**KEY CONCEPTS:**
- **ReLU (Rectified Linear Unit)** = Activation function that zeros out negatives: f(x) = max(0, x)
- **Non-linearity** = ReLU allows networks to learn complex patterns (without it, stacking layers = one linear operation)
- **Max Pooling** = Takes the maximum value in each 2×2 window, reducing spatial size by half
- **Encoder block** = Conv → ReLU → Pool - the repeating unit that compresses images



In [ ]:
# Apply each operation separately to track transformations
conv_layer = nn.Conv2d(1, 16, kernel_size=3, padding=1)
relu_layer = nn.ReLU()
pool_layer = nn.MaxPool2d(2)

In [ ]:
# Step 1: Convolution
conv_out = conv_layer(img_tensor)

# Step 2: ReLU activation
relu_out = relu_layer(conv_out)

# Step 3: Max pooling
pool_out = pool_layer(relu_out)

In [ ]:
print(f"  Input:      {img_tensor.shape}  → 15×15, 1 channel")
print(f"  After Conv: {conv_out.shape} → 15×15, 16 channels")
print(f"  After ReLU: {relu_out.shape} → 15×15, 16 channels")
print(f"  After Pool: {pool_out.shape}   → 7×7, 16 channels")

In [ ]:
# Visualize the 4-step pipeline
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

# Step 0: Input
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input\n15×15×1', fontweight='bold', fontsize=11)
axes[0].axis('off')

# Step 1: After convolution (channel 0 shown)
axes[1].imshow(conv_out[0, 0].detach(), cmap='RdBu_r')
axes[1].set_title('After Conv\n15×15×16\n(has negatives)', fontweight='bold', fontsize=11)
axes[1].axis('off')

# Step 2: After ReLU (channel 0 shown)
axes[2].imshow(relu_out[0, 0].detach(), cmap='viridis')
axes[2].set_title('After ReLU\n15×15×16\n(no negatives)', fontweight='bold', fontsize=11)
axes[2].axis('off')

# Step 3: After max pooling (channel 0 shown)
axes[3].imshow(pool_out[0, 0].detach(), cmap='viridis')
axes[3].set_title('After MaxPool\n7×7×16\n(downsampled)', fontweight='bold', fontsize=11)
axes[3].axis('off')

plt.suptitle('Encoder Block Pipeline: Conv → ReLU → MaxPool', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4. Full Encoder (Stacking Blocks)

**KEY CONCEPTS:**
- **Stacking blocks** = Repeating the Conv+ReLU+Pool pattern multiple times
- **Progressive compression** = Each block halves the spatial size (15→7→3)
- **Feature hierarchy** = Early layers detect simple patterns, deeper layers detect complex patterns


In [ ]:
encoder = nn.Sequential(
    # BLOCK 1: 15×15×1 → 7×7×16
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    # BLOCK 2: 7×7×16 → 3×3×32
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)) 

In [ ]:
# Extract outputs after each block
block1 = nn.Sequential(*list(encoder.children())[:5])  
block2 = encoder  # Full encoder = Block 1 + Block 2

output_block1 = block1(img_tensor)
output_block2 = block2(img_tensor)

print(f"  Input:         {img_tensor.shape}  → 15×15, 1 channel")
print(f"  After Block 1: {output_block1.shape}  → 7×7, 16 channels")
print(f"  After Block 2: {output_block2.shape}   → 3×3, 32 channels")

In [ ]:
# Visualize: Input → Block 1 → Block 2
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Input image
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input\n15×15×1\n(225 pixels)', fontweight='bold', fontsize=12)
axes[0].axis('off')

# After Block 1 (show channel 0)
axes[1].imshow(output_block1[0, 0].detach(), cmap='viridis')
axes[1].set_title('After Block 1\n7×7×16\n(49 pixels, 16 channels)', fontweight='bold', fontsize=12)
axes[1].axis('off')

# After Block 2 (show channel 0)
axes[2].imshow(output_block2[0, 0].detach(), cmap='viridis')
axes[2].set_title('After Block 2\n3×3×32\n(9 pixels, 32 channels)', fontweight='bold', fontsize=12)
axes[2].axis('off')

plt.suptitle('Encoder: Progressive Spatial Compression + Channel Expansion', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📉 Spatial shrinks: 225 → 49 → 9 pixels (25× compression)")
print("📈 Channels grow: 1 → 16 → 32 features (32× richer)")

### 5. Decoder (Reversing the Encoder)

**KEY CONCEPTS:**
- **Decoder** = The upsampling path that expands compressed features back to full resolution
- **Transposed convolution (ConvTranspose2d)** = Learnable upsampling that doubles spatial size
- **Mirror structure** = Decoder reverses encoder: spatial grows, channels shrink



In [ ]:
decoder = nn.Sequential(
    
    # BLOCK 1: 3×3×32 → 6×6×16
    nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),  # Upsample 2×
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),

    # BLOCK 2: 6×6×16 → 12×12×1
    nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2),   # Upsample 2×
    nn.ReLU(),
    nn.Conv2d(8, 1, kernel_size=3, padding=1),
    nn.Sigmoid()  # Output probabilities between 0 and 1
)

encoder_out = output_block2    
decoder_out = decoder(encoder_out)    

In [ ]:
print(f"  Encoder output: {encoder_out.shape}   → 3×3, 32 channels")
print(f"  After Block 1:  (1, 16, 6, 6)    → 6×6, 16 channels")
print(f"  After Block 2:  {decoder_out.shape}  → 12×12, 1 channel")
print(f"\n  📈 Spatial: 3×3 → 6×6 → 12×12 (16× larger)")
print(f"  📉 Channels: 32 → 16 → 1 (back to single output)")

In [ ]:
# Extract intermediate decoder outputs
decoder_block1 = nn.Sequential(*list(decoder.children())[:4])
decoder_full = decoder

output_decoder1 = decoder_block1(encoder_out)
output_decoder2 = decoder_full(encoder_out)

# Visualize: Compressed → Block 1 → Block 2
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Encoder output (compressed)
axes[0].imshow(encoder_out[0, 0].detach(), cmap='viridis')
axes[0].set_title('Encoder Output\n3×3×32\n(compressed)', fontweight='bold', fontsize=12)
axes[0].axis('off')

# After decoder block 1
axes[1].imshow(output_decoder1[0, 0].detach(), cmap='viridis')
axes[1].set_title('After Decoder Block 1\n6×6×16\n(expanding)', fontweight='bold', fontsize=12)
axes[1].axis('off')

# After decoder block 2 (final output)
axes[2].imshow(output_decoder2[0, 0].detach(), cmap='viridis')
axes[2].set_title('Final Output\n12×12×1\n(reconstructed)', fontweight='bold', fontsize=12)
axes[2].axis('off')

plt.suptitle('Decoder: Progressive Spatial Expansion + Channel Reduction', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. The Complete U-Net Model


**KEY CONCEPTS:**
- **nn.Module** = PyTorch's base class for building custom networks
- **Class-based architecture** = Reusable, modular, professional structure
- **U-Net** = Encoder + Bottleneck + Decoder forming a U-shape


In [ ]:
# Build complete U-Net: Encoder + Bottleneck + Decoder
model = nn.Sequential(
    
    # === ENCODER ===
    # Block 1: 15×15×1 → 7×7×16
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # Block 2: 7×7×16 → 3×3×32
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    # === BOTTLENECK ===
    # 3×3×32 → 3×3×64
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.ReLU(),

     
    # === DECODER ===
    # Block 1: 3×3×64 → 7×7×32
    nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    
    # Block 2: 7×7×32 → 15×15×16
    nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),

        
    # === OUTPUT ===
    nn.Conv2d(16, 1, kernel_size=1),
    nn.Sigmoid()

)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Complete U-Net assembled!")
print(f"  Parameters: {total_params:,}")

final_output = model(img_tensor)

In [ ]:
print("Complete U-Net:")
print(f"  Input:  {img_tensor.shape}  → 15×15 image")
print(f"  Output: {final_output.shape} → 12×12 probability map")
print(f"  ✅ Same size: Ready for segmentation!")

In [ ]:
class UNet(nn.Module):
    """
    U-Net for image segmentation.
    Input: (batch, 1, H, W) - grayscale images
    Output: (batch, 1, H, W) - probability maps
    """
    
    def __init__(self):
        super(UNet, self).__init__()
        
        # Encoder blocks
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Decoder blocks
        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        self.upconv2 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Output layer
        self.output = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        """Forward pass through the network"""
        # Encoder
        enc1 = self.enc1(x)
        enc1_pool = self.pool1(enc1)
        
        enc2 = self.enc2(enc1_pool)
        enc2_pool = self.pool2(enc2)
        
        # Bottleneck
        bottleneck = self.bottleneck(enc2_pool)
        
        # Decoder
        dec1_up = self.upconv1(bottleneck)
        dec1 = self.dec1(dec1_up)
        
        dec2_up = self.upconv2(dec1)
        dec2 = self.dec2(dec2_up)
        
        # Output
        out = self.output(dec2)
        return out

print("✓ UNet class defined")

### 7. Training the Model

**KEY CONCEPTS:**
- **Training** = Adjusting weights through repeated examples to minimize error
- **Loss function (BCE)** = Measures how wrong predictions are (lower = better)
- **Optimizer (Adam)** = Algorithm that adjusts weights based on gradients
- **Epoch** = One complete pass through the training data
- **Backpropagation** = Computing gradients and updating weights


In [ ]:
# Create a fresh U-Net with random weights
model = UNet()

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Fresh U-Net created")
print(f"  Parameters: {total_params:,} (all random)")




# Create training image (16×16 for clean downsampling)
train_img = torch.zeros(16, 16)
train_img[:, 8] = 1.0              # Vertical line
train_img[3:7, 11:15] = 1.0        # Top-right square
train_img[10:14, 2:6] = 1.0        # Bottom-left square
train_img_tensor = train_img.unsqueeze(0).unsqueeze(0)


# Create target labels (detect ALL white pixels)
train_labels = torch.zeros(16, 16)
train_labels[:, 8] = 1.0           # Line
train_labels[3:7, 11:15] = 1.0     # Top square
train_labels[10:14, 2:6] = 1.0     # Bottom square
train_labels_tensor = train_labels.unsqueeze(0).unsqueeze(0)


# Loss function: Binary Cross-Entropy
criterion = nn.BCELoss()

# Optimizer: Adam (adjusts weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [ ]:
# Training loop
NUM_EPOCHS = 100
losses = []
accuracies = []


for epoch in range(NUM_EPOCHS):
    # Forward pass: make prediction
    output = model(train_img_tensor)
    
    # Calculate loss: how wrong are we?
    loss = criterion(output, train_labels_tensor)
    
    # Backward pass: compute gradients
    optimizer.zero_grad()
    loss.backward()
    
    # Update weights
    optimizer.step()
    
    # Track metrics
    pred_binary = (output > 0.5).float()
    accuracy = (pred_binary == train_labels_tensor).float().mean().item() * 100
    
    losses.append(loss.item())
    accuracies.append(accuracy)
    
    # Print progress
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {loss.item():.4f} | Accuracy: {accuracy:5.2f}%")


In [ ]:
print(f"\n✓ Training complete!")
print(f"  Initial accuracy: {accuracies[0]:.1f}%")
print(f"  Final accuracy:   {accuracies[-1]:.1f}%")
print(f"  Improvement:      {accuracies[-1] - accuracies[0]:.1f} percentage points")

In [ ]:
# Get final prediction
model.eval()
with torch.no_grad():
    final_output = model(train_img_tensor)

# Create 4-panel visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Panel 1: Input
axes[0].imshow(train_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input\n(what network sees)', fontweight='bold', fontsize=11)
axes[0].axis('off')

# Panel 2: Target
axes[1].imshow(train_labels, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title('Target\n(what we want)', fontweight='bold', fontsize=11)
axes[1].axis('off')

# Panel 3: Prediction
axes[2].imshow(final_output.squeeze().detach(), cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title(f'Prediction\n({accuracies[-1]:.1f}% accurate)', fontweight='bold', fontsize=11)
axes[2].axis('off')

# Panel 4: Accuracy curve
axes[3].plot(range(1, NUM_EPOCHS+1), accuracies, linewidth=2, color='green')
axes[3].set_xlabel('Epoch', fontweight='bold')
axes[3].set_ylabel('Accuracy (%)', fontweight='bold')
axes[3].set_title('Learning Progress', fontweight='bold', fontsize=11)
axes[3].set_ylim([40, 105])
axes[3].axhline(y=100, color='red', linestyle='--', alpha=0.3, label='Perfect')
axes[3].grid(alpha=0.3)
axes[3].legend()

plt.suptitle(f'Trained U-Net: {accuracies[-1]:.1f}% Pixel Accuracy', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**⚠️ Important limitation:** We trained on just ONE image and tested on that SAME image - the network memorized this specific pattern, not general concepts. In real applications, we need

### 8. Testing Generalization

**KEY CONCEPTS:**
- **Generalization** = Performing well on new, unseen data
- **Overfitting** = Memorizing training data without learning general patterns
- **Train vs Test split** = Training data teaches, test data measures generalization
- **Generalization gap** = Difference between train and test accuracy


In [ ]:
# Create fresh U-Net with random weights
model = UNet()

print("✓ Fresh U-Net created for generalization test")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TRAINING IMAGE (what network learns from)
train_img = torch.zeros(16, 16)
train_img[:, 8] = 1.0              # Vertical line
train_img[3:7, 11:15] = 1.0        # Top-right square
train_img[10:14, 2:6] = 1.0        # Bottom-left square
train_img_tensor = train_img.unsqueeze(0).unsqueeze(0)

train_labels = torch.zeros(16, 16)
train_labels[:, 8] = 1.0
train_labels[3:7, 11:15] = 1.0
train_labels[10:14, 2:6] = 1.0
train_labels_tensor = train_labels.unsqueeze(0).unsqueeze(0)


# TEST IMAGE (NEVER seen during training!)
test_img = torch.zeros(16, 16)
test_img[5, :] = 1.0               # HORIZONTAL line (different!)
test_img[2:5, 2:5] = 1.0          # Top-left square (different position!)
test_img[10:13, 10:13] = 1.0      # Bottom-right square (different position!)
test_img_tensor = test_img.unsqueeze(0).unsqueeze(0)

test_labels = torch.zeros(16, 16)
test_labels[5, :] = 1.0
test_labels[2:5, 2:5] = 1.0
test_labels[10:13, 10:13] = 1.0
test_labels_tensor = test_labels.unsqueeze(0).unsqueeze(0)


In [ ]:
# Training setup
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

NUM_EPOCHS = 200


# Training loop with train/test tracking
train_accuracies = []
test_accuracies = []

print(f"\n🏋️ Training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    # TRAIN on training image
    model.train()
    output = model(train_img_tensor)
    loss = criterion(output, train_labels_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Evaluate on TRAINING image
    with torch.no_grad():
        pred_train = (output > 0.5).float()
        train_acc = (pred_train == train_labels_tensor).float().mean().item() * 100
    
    # Evaluate on TEST image (unseen!)
    model.eval()
    with torch.no_grad():
        output_test = model(test_img_tensor)
        pred_test = (output_test > 0.5).float()
        test_acc = (pred_test == test_labels_tensor).float().mean().item() * 100
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    # Print progress
    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train: {train_acc:5.1f}% | Test: {test_acc:5.1f}%")


In [ ]:
print(f"  Final train accuracy: {train_accuracies[-1]:.1f}% (memorization)")
print(f"  Final test accuracy:  {test_accuracies[-1]:.1f}% (generalization)")
print(f"  Generalization gap:   {train_accuracies[-1] - test_accuracies[-1]:.1f} percentage points")

In [ ]:
# Get final predictions
model.eval()
with torch.no_grad():
    final_train_pred = model(train_img_tensor)
    final_test_pred = model(test_img_tensor)

# Create 2-row comparison visualization
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# === ROW 1: TRAINING DATA ===
axes[0, 0].imshow(train_img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Training Input', fontweight='bold', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(train_labels, cmap='RdYlGn', vmin=0, vmax=1)
axes[0, 1].set_title('Training Target', fontweight='bold', fontsize=11)
axes[0, 1].axis('off')

axes[0, 2].imshow(final_train_pred.squeeze(), cmap='RdYlGn', vmin=0, vmax=1)
axes[0, 2].set_title(f'Training Prediction\n{train_accuracies[-1]:.1f}% accurate', 
                     fontweight='bold', fontsize=11)
axes[0, 2].axis('off')

# Learning curves
axes[0, 3].plot(range(1, NUM_EPOCHS+1), train_accuracies, linewidth=2, color='blue', label='Train')
axes[0, 3].plot(range(1, NUM_EPOCHS+1), test_accuracies, linewidth=2, color='orange', label='Test')
axes[0, 3].set_xlabel('Epoch', fontweight='bold')
axes[0, 3].set_ylabel('Accuracy (%)', fontweight='bold')
axes[0, 3].set_title('Learning Progress', fontweight='bold', fontsize=11)
axes[0, 3].set_ylim([0, 105])
axes[0, 3].axhline(y=100, color='red', linestyle='--', alpha=0.3)
axes[0, 3].grid(alpha=0.3)
axes[0, 3].legend()

# === ROW 2: TEST DATA (UNSEEN) ===
axes[1, 0].imshow(test_img, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Test Input\n(NEVER SEEN!)', fontweight='bold', fontsize=11, color='red')
axes[1, 0].axis('off')

axes[1, 1].imshow(test_labels, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 1].set_title('Test Target', fontweight='bold', fontsize=11)
axes[1, 1].axis('off')

axes[1, 2].imshow(final_test_pred.squeeze(), cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_title(f'Test Prediction\n{test_accuracies[-1]:.1f}% accurate', 
                     fontweight='bold', fontsize=11)
axes[1, 2].axis('off')

# Summary panel
gen_status = "✅ Good!" if test_accuracies[-1] > 70 else "❌ Poor"
summary = f"""
GENERALIZATION TEST

Trained on:
- Vertical line (x=8)
- Specific square positions

Tested on:
- Horizontal line (y=5)  
- Different square positions

Train Accuracy: {train_accuracies[-1]:.1f}%
Test Accuracy:  {test_accuracies[-1]:.1f}%

Gap: {train_accuracies[-1] - test_accuracies[-1]:.1f}% {gen_status}
"""

axes[1, 3].text(0.05, 0.5, summary, fontsize=10, verticalalignment='center', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1, 3].set_xlim([0, 1])
axes[1, 3].set_ylim([0, 1])
axes[1, 3].axis('off')

plt.suptitle(f'Generalization Test: Train={train_accuracies[-1]:.1f}% | Test={test_accuracies[-1]:.1f}%', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()